# Hybrid Fusion: Combining Transformer and Lexicon-Based Scores

This notebook combines the DistilBERT baseline outputs with snippet/lexicon scores
to create a hybrid sentiment representation. The goal is to balance transformer confidence
with rule-based interpretability.

In [1]:
# 1. Setup
import pandas as pd

# Load transformer baseline predictions (5-class)
baseline = pd.read_csv("../data/processed/distilbert_baseline_5class.csv")

# Load snippet/lexicon scores (5-class)
snippet = pd.read_csv("../data/processed/snippet_scores_5class.csv")

print(f"✅ Loaded Baseline predictions: {baseline.shape}")
print(f"✅ Loaded Snippet scores: {snippet.shape}")

✅ Loaded Baseline predictions: (14166, 6)
✅ Loaded Snippet scores: (9, 6)


In [2]:
# 2. Quick sanity checks
print("Baseline columns:", baseline.columns.tolist())
print("Snippet columns:", snippet.columns.tolist())

print("Companies:", snippet['company'].unique())
print("Years:", snippet['year'].unique())

Baseline columns: ['company', 'year', 'sentence', 'label', 'score', 'sentiment_5class']
Snippet columns: ['company', 'year', 'pos', 'neg', 'score', 'lexicon_5class']
Companies: ['Google' 'HSBC' 'Nestle']
Years: [2022 2023 2024]


In [3]:
# 3. Aggregate Baseline predictions to document-level
# (since snippet is at document-level, we need to match granularity)

bert_doc = (
    baseline.groupby(["company", "year"])
    .agg({
        "score": "mean"  # average confidence score across sentences
    })
    .reset_index()
    .rename(columns={"score": "bert_mean_score"})
)

print("✅ Aggregated Baseline predictions to doc-level:", bert_doc.shape)

✅ Aggregated Baseline predictions to doc-level: (9, 3)


In [10]:
# 4. Merge both sources
hybrid_df = pd.merge(snippet, bert_doc, on=["company", "year"], how="inner")

print("✅ Hybrid DataFrame created:", hybrid_df.shape)
print(hybrid_df.head())

✅ Hybrid DataFrame created: (9, 7)
  company  year  pos  neg  score lexicon_5class  bert_mean_score
0  Google  2022  140  107     33        NEUTRAL         0.898745
1  Google  2023  951  654    297  VERY POSITIVE         0.937662
2  Google  2024  689  705    -16        NEUTRAL         0.927499
3    HSBC  2022  818  672    146       POSITIVE         0.942709
4    HSBC  2023  829  785     44        NEUTRAL         0.938589


In [16]:
def hybrid_to_5class(score, high=70, low=20):
    if score >= high:
        return "VERY POSITIVE"
    elif score >= low:
        return "POSITIVE"
    elif score <= -high:
        return "VERY NEGATIVE"
    elif score <= -low:
        return "NEGATIVE"
    else:
        return "NEUTRAL"

In [17]:
# 5. Simple fusion strategy
# Example: weighted average (70% Baseline, 30% Lexicon)
hybrid_df["hybrid_score"] = (
    0.7 * hybrid_df["bert_mean_score"] + 0.3 * hybrid_df["score"]
)

# Apply 5-class mapping directly on the hybrid_df we just created
hybrid_df['hybrid_5class'] = hybrid_df['hybrid_score'].apply(lambda x: hybrid_to_5class(x))

# Save as 5-class CSV
out_path = "../data/processed/hybrid_scores_5class.csv"
hybrid_df.to_csv(out_path, index=False)
print(f"✅ Saved hybrid 5-class scores to {out_path}")
print(hybrid_df['hybrid_5class'].value_counts().to_string())

✅ Saved hybrid 5-class scores to ../data/processed/hybrid_scores_5class.csv
hybrid_5class
NEUTRAL          3
VERY POSITIVE    3
POSITIVE         2
NEGATIVE         1
